# 🚓 UK Crime — Where, What & How it's resolved
**Author:** Robert Maszkiewski · **Toolkit:** Python (pandas, matplotlib, calamine) · **Data:** Home Office & ONS open data (England & Wales, year ending March 2025)

*Follows the case-study checklist:* **Introduction → Problems → Solutions → Conclusion → Next steps**, *with the 6 phases (Ask · Prepare · Process · Analyze · Share · Act) embedded.*

> ✏️ **About me:** Aspiring data analyst (Google Data Analytics certificate) who turns messy, real-world data into clear, defensible decisions with SQL, Python and dashboards. Open to junior data-analyst roles.
> 📫 robertmaszkiewski@gmail.com · GitHub: https://github.com/robertmaszkiewski · Portfolio: https://rmportfolio.co.uk · LinkedIn: https://www.linkedin.com/in/robert-maszkiewski

**Live interactive dashboard for this study → https://rmportfolio.co.uk/case-studies/uk-crime.html**

## 1. Introduction *(Ask)*

Crime headlines usually quote **raw counts**, which mostly tells you where the most *people* live. The useful questions are sharper:

1. **Where** is crime highest **per head of population**? *(not just raw totals)*
2. **What** kinds of crime drive the totals?
3. **When** — which way has the trend moved?
4. **How** are crimes resolved — what share end in a charge?

**Scope & honesty (this matters for trust):**
- **Recorded crime ≠ actual crime.** These are offences *recorded by police*; they depend on what's reported and how it's logged. Part of the 2013–2019 rise reflects improved recording, not only more crime. The Crime Survey (CSEW) measures victimisation differently.
- **England & Wales only** (43 territorial forces). Scotland & NI use separate systems.
- **Fraud is recorded centrally** (Action Fraud), not by local forces → excluded from per-area rates, reported separately.
- **Rates use ONS mid-2024 resident population** → distorts commuter hubs (City of London is flagged, not headlined).
- **No offender demographics exist** in UK open crime data (no nationality/ethnicity of offenders), so this study makes **no** such claims — they would not be supportable from the data.

In [ ]:
# --- Setup ---
import os, io, glob, sys, warnings, subprocess
warnings.filterwarnings("ignore")
try:
    from python_calamine import CalamineWorkbook
except Exception:
    subprocess.run([sys.executable,"-m","pip","install","-q","python-calamine"], check=False)
    from python_calamine import CalamineWorkbook
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize":(10,4.8),"axes.spines.top":False,"axes.spines.right":False,
                     "axes.grid":True,"grid.alpha":.25,"font.size":11})
NAVY,BLUE,RED,GREY = "#16335c","#2f6db0","#c0392b","#9aa3b2"

# Official open-data sources (Open Government Licence v3.0)
URLS = {
 "crime":"https://assets.publishing.service.gov.uk/media/69e634ef84ac36aaf2cc92e2/prc-pfa-mar2013-onwards-tables-230426.ods",
 "pop":"https://www.ons.gov.uk/file?uri=/peoplepopulationandcommunity/populationandmigration/populationestimates/adhocs/3194populationestimatesforpoliceforceareasinenglandandwalesbysingleyearofageandsexmid1991tomid2024/policeforceareas1991to2024.xlsx",
 "outcomes":"https://assets.publishing.service.gov.uk/media/69e1dbc46c6ccf244be42eca/prc-outcomes-open-data-mar2025-tables-230426.xlsx",
}
def fetch(key, fname):
    "Use a Kaggle-input copy if present, else download (needs Internet ON in Kaggle settings)."
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if hits: print("using local:", hits[0]); return hits[0]
    if os.path.exists(fname): return fname
    import urllib.request
    print("downloading", fname, "...")
    urllib.request.urlretrieve(URLS[key], fname)
    return fname
CRIME = fetch("crime","prc-pfa.ods")
print("ready:", CRIME)

## 2. Problems *(the analytical questions)* & 3. Prepare

The Home Office **Police Force Area** table ships one sheet per financial year (2012/13 → 2025/26), each row = *force × quarter × offence code × count*. First we load every year, then sanity-check the latest data.

In [ ]:
# --- Load all year sheets (first 8 columns are consistent across years) ---
COLS=['Financial Year','Financial Quarter','Force Name','Offence Description','Offence Group','Offence Subgroup','Offence Code','Number of Offences']
wb=CalamineWorkbook.from_path(CRIME)
years=[s for s in wb.sheet_names if s[0].isdigit()]
cr=pd.concat([pd.DataFrame([r[:8] for r in wb.get_sheet_by_name(s).to_python()[1:]],columns=COLS) for s in years],ignore_index=True)
cr['n']=pd.to_numeric(cr['Number of Offences'],errors='coerce').fillna(0)
cr['FY']=cr['Financial Year'].astype(str).str.strip()
cr['Force']=cr['Force Name'].astype(str).str.replace('&','and').str.strip()
cr['Group']=cr['Offence Group'].astype(str).str.strip()
# 6 of the 'forces' are central fraud bodies / BTP — not geographic areas
NONGEO={'Action Fraud','CIFAS','Cifas','Financial Fraud Action UK','UK Finance','British Transport Police'}
cr['territorial']=~cr['Force Name'].astype(str).str.strip().isin(NONGEO)
print("rows:",len(cr)," territorial forces:",cr.loc[cr['territorial'],'Force'].nunique())
print("offence groups:",sorted(cr['Group'].unique()))
# which years are complete? (4 quarters)
q=cr.groupby('FY')['Financial Quarter'].nunique()
print("\nquarters per recent year:\n",q.tail(5))

**Decision:** the most recent sheet (2025/26) is **incomplete** (not all four quarters), so the headline "latest 12 months" uses the last **complete** financial year, **2024/25 (year ending March 2025)**. This is the kind of check that stops you publishing a half-year as if it were a full one.

In [ ]:
FY='2024/25'
fy=cr[cr['FY']==FY]
terr_nf=fy[(fy['territorial'])&(fy['Group']!='Fraud offences')]   # geographic, comparable base
recon={'all bodies':int(fy['n'].sum()),
       'territorial excl fraud':int(terr_nf['n'].sum()),
       'fraud (central)':int(fy[fy['Group']=='Fraud offences']['n'].sum())}
for k,v in recon.items(): print(f"{k:24s}: {v:,}")

## 4. Process — population for per-capita rates

To compare areas fairly we divide by **resident population** (ONS mid-2024). We sum the single-year-of-age/sex columns to a total per Police Force Area, then join on force name.

In [ ]:
POP=fetch("pop","pfa-pop.xlsx")
pw=CalamineWorkbook.from_path(POP); rows=pw.get_sheet_by_name("Mid-2021 to Mid-2024").to_python()
hdr=rows[3]; pop=pd.DataFrame(rows[4:],columns=hdr)
age=hdr[3:]
for c in age: pop[c]=pd.to_numeric(pop[c],errors='coerce')
pop['Year']=pd.to_numeric(pop['Year'],errors='coerce')
pop['Population']=pop[age].sum(axis=1)
pop24=pop[pop['Year']==2024][['PFA 2023 Name','Population']].copy()
pop24['Force']=pop24['PFA 2023 Name'].str.replace('&','and').str.strip()
print("E&W population (mid-2024): {:,}".format(int(pop24['Population'].sum())))
bf=terr_nf.groupby('Force')['n'].sum().reset_index(name='crimes').merge(pop24[['Force','Population']],on='Force',how='left')
bf['rate']=(bf['crimes']/bf['Population']*1000).round(1)
print("unmatched forces:",bf[bf['Population'].isna()]['Force'].tolist())
print("E&W overall rate / 1,000:",round(terr_nf['n'].sum()/pop24['Population'].sum()*1000,1))

## 5. Analyze & Share

### 5.1 Where — crime rate per 1,000 residents
Raw counts peak in London (most people); **per head**, the highest rates are northern metros. **City of London** (≈9k residents, huge daytime population) is a denominator artifact — shown in red, not used as the headline.

In [ ]:
d=bf.sort_values('rate',ascending=False).head(15)
col=[RED if f=='London, City of' else BLUE for f in d['Force']]
fig,ax=plt.subplots(figsize=(10,6)); ax.barh(d['Force'][::-1],d['rate'][::-1],color=col[::-1])
ax.set_xlabel("Crimes per 1,000 residents (2024/25, excl. fraud)"); ax.set_title("Where: highest crime RATES per 1,000")
plt.tight_layout(); plt.show()
print("Top by raw count:\n",bf.sort_values('crimes',ascending=False).head(3)[['Force','crimes','rate']].to_string(index=False))

In [ ]:
# Optional choropleth (needs geopandas + boundary GeoJSON; safe to skip)
try:
    import geopandas as gpd
    gj="pfa.geojson"
    if not os.path.exists(gj):
        import urllib.request
        urllib.request.urlretrieve("https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Police_Force_Areas_December_2023_EW_BUC/FeatureServer/0/query?where=1%3D1&outFields=PFA23NM&outSR=4326&f=geojson", gj)
    g=gpd.read_file(gj); g['Force']=g['PFA23NM'].str.replace('&','and').str.strip()
    g=g.merge(bf[['Force','rate']],on='Force',how='left')
    ax=g.plot(column='rate',cmap='Blues',edgecolor='white',linewidth=.4,legend=True,figsize=(7,8),missing_kwds={'color':'#eee'})
    ax.set_title("Crime rate per 1,000 residents by police force area (2024/25)"); ax.axis('off'); plt.tight_layout(); plt.show()
except Exception as e:
    print("Choropleth skipped:", e)

### 5.2 What — offences by type
Violence and theft dominate; fraud (central) is reported separately.

In [ ]:
GROUPS9=['Violence against the person','Theft offences','Criminal damage and arson','Public order offences','Sexual offences','Drug offences','Miscellaneous crimes against society','Robbery','Possession of weapons offences']
bg=terr_nf.groupby('Group')['n'].sum().reindex(GROUPS9).sort_values()
fig,ax=plt.subplots(); ax.barh(bg.index,bg.values/1e6,color=NAVY); ax.set_xlabel("Million offences (2024/25)")
ax.set_title("What: offences by type (England & Wales, excl. fraud)"); plt.tight_layout(); plt.show()
for g_,v in bg.sort_values(ascending=False).items(): print(f"{g_:42s} {int(v):>10,}  {v/bg.sum()*100:4.1f}%")

### 5.3 When — the trend
Risen since 2012/13 (partly improved recording), COVID dip in 2020/21, peak 2022/23, easing since. (2025/26 excluded — incomplete.)

In [ ]:
yrs=[f for f in sorted(cr['FY'].unique()) if f!='2025/26']
tr=cr[(cr['territorial'])&(cr['Group']!='Fraud offences')&(cr['FY'].isin(yrs))].groupby('FY')['n'].sum()
fig,ax=plt.subplots(); ax.plot(list(tr.index),tr.values/1e6,marker='o',color=BLUE,lw=2)
ax.set_ylabel("Million offences"); ax.set_title("When: recorded crime, territorial forces excl. fraud"); plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.show()
# Shoplifting — a sharp recent rise
shop=cr[(cr['territorial'])&(cr['FY'].isin(yrs))&(cr['Offence Subgroup'].astype(str).str.contains('Shoplifting',case=False))].groupby('FY')['n'].sum()
fig,ax=plt.subplots(); ax.plot(list(shop.index),shop.values/1e3,marker='o',color=RED,lw=2); ax.set_ylabel("Thousand offences"); ax.set_title("Shoplifting — sharp recent rise"); plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.show()
print("shoplifting:",{k:int(v) for k,v in shop.tail(3).items()})

### 5.4 How it's resolved — outcomes
The Outcomes open-data file records, for crimes logged in the year, what outcome they reached.

In [ ]:
OUT=fetch("outcomes","prc-outcomes.xlsx")
ow=CalamineWorkbook.from_path(OUT)
sheet=[s for s in ow.sheet_names if 'Outcomes open data' in s][0]
orows=ow.get_sheet_by_name(sheet).to_python(); oh=orows[0]
oc=pd.DataFrame(orows[1:],columns=oh)
cntcol=[c for c in oh if 'Outcomes for offences' in c][0]
oc['n']=pd.to_numeric(oc[cntcol],errors='coerce').fillna(0)
og=oc.groupby('Outcome Group')['n'].sum().sort_values(ascending=False); tot=og.sum()
print(f"Total outcomes {int(tot):,}")
print(f"Charged/Summonsed: {int(og.get('Charged/Summonsed',0)):,}  ({og.get('Charged/Summonsed',0)/tot*100:.1f}%)")
o=og.head(8)[::-1]; col=[RED if 'Charged' in i else NAVY for i in o.index]
fig,ax=plt.subplots(figsize=(10,5.5)); ax.barh([i[:46] for i in o.index],o.values/1e3,color=col); ax.set_xlabel("Thousand outcomes")
ax.set_title("Outcomes: only ~8.6% of crimes end in a charge/summons"); plt.tight_layout(); plt.show()

In [ ]:
# Charge rate by offence type
g=oc[oc['Offence Group'].isin(GROUPS9)].groupby('Offence Group')['n'].sum()
ch=oc[(oc['Outcome Group']=='Charged/Summonsed')&(oc['Offence Group'].isin(GROUPS9))].groupby('Offence Group')['n'].sum()
cgr=(ch/g*100).round(1).reindex(GROUPS9).sort_values()
fig,ax=plt.subplots(); ax.barh(cgr.index,cgr.values,color=BLUE); ax.set_xlabel("% charged / summonsed"); ax.set_title("Charge rate by offence type (2024/25)")
plt.tight_layout(); plt.show()
print(cgr.sort_values(ascending=False).to_string())

## 6. Conclusion & Next steps

**Key takeaways**
- **Raw counts mislead.** London records the most crime because it has the most people; **per 1,000 residents** the highest rates are in Cleveland, West Yorkshire and Greater Manchester.
- **Two types dominate:** violence (~37%) and theft (~33%); **shoplifting is up ~54% in two years**.
- **Most crimes go unresolved:** only **~8.6%** end in a charge/summons; ~40% close with no suspect identified.
- **Lesson:** the numbers are only as honest as their caveats — population denominators, central fraud recording, and "recorded ≠ actual" all change the read.

**Next steps**
1. Add a **daytime-population** denominator so commuter hubs are comparable.
2. Drill into **street-level hotspots** (data.police.uk lat/long) for the top forces.
3. Join to **deprivation (IMD)** to test how much area variation socio-economics explains.
4. Track **shoplifting & charge rates quarterly** as the headline metrics to watch.

*Data: Home Office "Police recorded crime and outcomes open data tables" and ONS population estimates — © Crown copyright, Open Government Licence v3.0. Full code & live dashboard: https://rmportfolio.co.uk/case-studies/uk-crime.html*